In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# ---- Clean + Stable Environment Setup for BioGPT Evaluation ----
!pip -q uninstall -y sentence-transformers
!pip -q install "jedi>=0.16"

# Upgrade build tools
!pip -q install --upgrade pip setuptools wheel

# Stable pandas for Colab
!pip -q install pandas==2.2.2

# HuggingFace + evaluation stack
!pip -q install \
    tokenizers==0.15.2 \
    transformers==4.39.3 \
    accelerate \
    datasets \
    bert-score \
    evaluate \
    sentencepiece \
    sacremoses \
    tqdm

# ---- Verify versions ----
import torch, pandas, transformers, tokenizers

print("torch:", torch.__version__)
print("pandas:", pandas.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import os

unseen_dir = "/content/drive/MyDrive/biomedical_text_generation/data/unseen/biogpt_unseen.jsonl"

# For BioGPT, prefer the raw unseen JSONL for evaluation/generation.
# BioGPT is decoder-only, so old BioBART/BioT5 tokenized seq2seq datasets may not fit.
TEST_DATASET_DIR = None

# Folder containing your fine-tuned BioGPT checkpoint
CHECKPOINT_DIR = "/content/drive/MyDrive/biomedical_text_generation/models/biogpt_title_text_gen_final"

OUTPUT_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biogpt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generation / eval params
MAX_INPUT_LEN  = 128
MAX_NEW_TOKENS = 895
BATCH_SIZE     = 8   # BioGPT can be heavier; increase to 8 if GPU memory allows

# BERTScore model
BERTSCORE_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
LANG = "en"

# BioGPT base model, useful if loading tokenizer separately
BASE_MODEL_NAME = "microsoft/biogpt"

In [ ]:
############## testing (BioGPT version) ############
import json

jsonl_path = unseen_dir

# Read dataset
with open(jsonl_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total rows:", len(lines))

# Load one example
ex = json.loads(lines[100])

print("Columns:", ex.keys())

# Adjust field names if needed (input/target vs text/title etc.)
INPUT_FIELD = "input"
TARGET_FIELD = "target"

print("\nExample input:", str(ex.get(INPUT_FIELD))[:500])
print("\nExample target:", str(ex.get(TARGET_FIELD))[:1500])

In [ ]:
import json
import numpy as np

jsonl_path = unseen_dir

input_lengths = []
target_lengths = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        example = json.loads(line)

        inp = example.get("input", "")
        tgt = example.get("target", "")

        input_lengths.append(len(inp.split()))
        target_lengths.append(len(tgt.split()))

print("Total examples:", len(input_lengths))

print("\n--- Input stats ---")
print("Mean:", np.mean(input_lengths))
print("Max :", np.max(input_lengths))
print("Min :", np.min(input_lengths))

print("\n--- Target stats ---")
print("Mean:", np.mean(target_lengths))
print("Max :", np.max(target_lengths))
print("Min :", np.min(target_lengths))

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

jsonl_path = unseen_dir

input_lengths = []
target_lengths = []

# Read JSONL file safely
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        example = json.loads(line)

        inp = example.get("input", "")
        tgt = example.get("target", "")

        input_lengths.append(len(inp.split()))
        target_lengths.append(len(tgt.split()))

print("Total examples:", len(input_lengths))

# ---- Input Histogram ----
plt.figure()
plt.hist(input_lengths, bins=50)
plt.title("Distribution of Input Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.axvline(np.mean(input_lengths), linestyle="dashed", label="Mean")
plt.legend()
plt.show()

# ---- Target Histogram ----
plt.figure()
plt.hist(target_lengths, bins=50)
plt.title("Distribution of Target Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.axvline(np.mean(target_lengths), linestyle="dashed", label="Mean")
plt.legend()
plt.show()

# ---- Descriptive Stats ----
print("\n--- Input Stats ---")
print("Mean   :", np.mean(input_lengths))
print("Median :", np.median(input_lengths))
print("95th % :", np.percentile(input_lengths, 95))
print("Max    :", np.max(input_lengths))

print("\n--- Target Stats ---")
print("Mean   :", np.mean(target_lengths))
print("Median :", np.median(target_lengths))
print("95th % :", np.percentile(target_lengths, 95))
print("Max    :", np.max(target_lengths))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)

# 🔥 CRITICAL FIX for BioGPT
tokenizer.padding_side = "left"

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(CHECKPOINT_DIR).to(device)
model.eval()

In [ ]:
from tqdm.auto import tqdm
import json
import torch

NUM_BEAMS = 4

preds = []
refs = []
inputs = []

# ---- Load raw examples ----
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        example = json.loads(line)

        inp = str(example.get("input", ""))
        tgt = str(example.get("target", ""))

        inputs.append(inp)
        refs.append(tgt)

print("Total examples:", len(inputs))
print("Sample input:", inputs[0][:500])
print("Sample ref:", refs[0][:500])


# ---- Prompt formatting for title -> abstract generation ----
def build_prompt(title):
    return f"Title: {title}\nAbstract:"


@torch.no_grad()
def generate_from_texts(texts):
    prompts = [build_prompt(t) for t in texts]

    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    input_len = input_ids.shape[1]

    # BioGPT max context length is usually 1024.
    # Total length = prompt tokens + generated tokens.
    max_context = getattr(model.config, "max_position_embeddings", 1024)

    # Safe dynamic generation cap for this batch
    safe_max_new_tokens = max_context - input_len - 1

    # Never exceed your requested MAX_NEW_TOKENS
    safe_max_new_tokens = min(MAX_NEW_TOKENS, safe_max_new_tokens)

    if safe_max_new_tokens <= 0:
        raise ValueError(
            f"Input length {input_len} is too long for model context {max_context}. "
            "Reduce MAX_INPUT_LEN."
        )

    gen_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=safe_max_new_tokens,
        num_beams=NUM_BEAMS,
        early_stopping=True,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Remove prompt tokens from causal LM output
    generated_only = gen_ids[:, input_len:]

    return tokenizer.batch_decode(generated_only, skip_special_tokens=True)


# ---- Generate predictions ----
for i in tqdm(range(0, len(inputs), BATCH_SIZE), desc="Generating with BioGPT"):
    batch_texts = inputs[i:i+BATCH_SIZE]
    preds.extend(generate_from_texts(batch_texts))

print("Generated:", len(preds))
print("\nREF:", refs[0][:300])
print("\nPRED:", preds[0][:300])

In [ ]:
idx = 1020  # change this to inspect different examples

print("Generated:", len(preds))

if idx < len(preds) and idx < len(refs):
    print("\nREF:\n", refs[idx][:3000])
    print("\nPRED:\n", preds[idx][:3000])
else:
    print(f"Index {idx} out of range.")

In [ ]:
from transformers import AutoTokenizer
from bert_score import score

BERTSCORE_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
NUM_LAYERS = 12

bs_tokenizer = AutoTokenizer.from_pretrained(BERTSCORE_MODEL, use_fast=True)

# ---- Optional BioGPT prediction cleanup ----
def clean_pred(p):
    p = str(p).strip()

    # Keep only first generated line/title
    p = p.split("\n")[0].strip()

    # Remove prompt artifacts if BioGPT echoes them
    for marker in ["Title:", "title:", "Abstract:", "abstract:"]:
        if marker in p:
            p = p.split(marker)[-1].strip()

    return p

preds_clean = [clean_pred(p) for p in preds]

def truncate_for_bertscore(texts, tokenizer, max_len=512):
    out = []

    for t in texts:
        if not isinstance(t, str):
            t = str(t)

        ids = tokenizer.encode(
            t,
            add_special_tokens=True,
            truncation=True,
            max_length=max_len
        )

        out.append(tokenizer.decode(ids, skip_special_tokens=True))

    return out

# Truncate both predictions and references
preds_trunc = truncate_for_bertscore(preds_clean, bs_tokenizer, max_len=512)
refs_trunc  = truncate_for_bertscore(refs,        bs_tokenizer, max_len=512)

# Sanity check
assert len(preds_trunc) == len(refs_trunc), "Mismatch between preds and refs!"

P, R, F1 = score(
    cands=preds_trunc,
    refs=refs_trunc,
    model_type=BERTSCORE_MODEL,
    num_layers=NUM_LAYERS,
    lang=LANG,
    verbose=True,
    rescale_with_baseline=False,
    batch_size=16  # reduce if OOM
)

print("\nBERTScore means (truncated to 512 tokens):")
print("  Precision:", float(P.mean()))
print("  Recall:   ", float(R.mean()))
print("  F1:       ", float(F1.mean()))

In [ ]:
import pandas as pd
import json
import os

jsonl_path = unseen_dir

inputs = []
refs = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        example = json.loads(line)
        inputs.append(str(example.get("input", "")))
        refs.append(str(example.get("target", "")))

# Safety check
assert len(inputs) == len(refs) == len(preds_clean) == len(P), "Length mismatch!"

out_df = pd.DataFrame({
    "input": inputs,
    "target": refs,
    "prediction_raw": preds,
    "prediction": preds_clean,
    "bertscore_P": P.cpu().numpy(),
    "bertscore_R": R.cpu().numpy(),
    "bertscore_F1": F1.cpu().numpy(),
})

csv_path = os.path.join(OUTPUT_DIR, "biogpt_test_with_bertscore.csv")
out_df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

summary = {
    "model": "BioGPT",
    "checkpoint_dir": CHECKPOINT_DIR,
    "test_source": jsonl_path,
    "generation": {
        "max_input_len": MAX_INPUT_LEN,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS,
        "batch_size": BATCH_SIZE
    },
    "bertscore_model": BERTSCORE_MODEL,
    "mean_precision": float(P.mean()),
    "mean_recall": float(R.mean()),
    "mean_f1": float(F1.mean()),
    "n_examples": int(len(out_df)),
}

json_path = os.path.join(OUTPUT_DIR, "biogpt_bertscore_title_text_gen.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", json_path)

out_df.head()

In [ ]:
# Single-model evaluation summary for BioGPT CSV
import pandas as pd
import numpy as np
import os
import json

BIOGPT_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biogpt/biogpt_test_with_bertscore.csv"

df = pd.read_csv(BIOGPT_PATH)

print("Rows in BioGPT file:", len(df))
print("Columns:", df.columns.tolist())

metrics = ["bertscore_P", "bertscore_R", "bertscore_F1"]

results = []

for metric in metrics:
    if metric not in df.columns:
        print(f"Skipping {metric}: column not found.")
        continue

    scores = df[metric].dropna().to_numpy(dtype=float)

    res = {
        "model": "BioGPT",
        "metric": metric,
        "n": int(len(scores)),
        "mean": float(np.mean(scores)),
        "std": float(np.std(scores, ddof=1)),
        "median": float(np.median(scores)),
        "min": float(np.min(scores)),
        "max": float(np.max(scores)),
        "q25": float(np.percentile(scores, 25)),
        "q75": float(np.percentile(scores, 75)),
    }

    results.append(res)

    print("\n" + "="*60)
    print(f"Metric: {metric}")
    print(f"Samples (N): {len(scores)}")
    print(f"Mean ± SD: {res['mean']:.4f} ± {res['std']:.4f}")
    print(f"Median:    {res['median']:.4f}")
    print(f"IQR:       [{res['q25']:.4f}, {res['q75']:.4f}]")
    print(f"Min/Max:   {res['min']:.4f} / {res['max']:.4f}")
    print("="*60)

results_df = pd.DataFrame(results)
results_df

In [ ]:
!pip -q install "jedi>=0.16"
!pip -q install -U pip setuptools wheel
!pip -q install "numpy==1.26.4" "pandas>=2.0"

# spaCy + scispaCy stack (no scipy)
!pip -q install "spacy==3.7.5" "thinc==8.2.5" "scispacy==0.5.4"

# SciSpaCy model
!pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_scibert-0.5.4.tar.gz

In [ ]:
!pip install spacy
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_lg-0.5.1.tar.gz

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import spacy
import scispacy
from scispacy.linking import EntityLinker

BIOGPT_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biogpt/biogpt_test_with_bertscore.csv"

df = pd.read_csv(BIOGPT_PATH)

# Prefer cleaned prediction if available
if "prediction" not in df.columns and "prediction_raw" in df.columns:
    df["prediction"] = df["prediction_raw"]

required_cols = ["input", "target", "prediction"]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

# -----------------------------
# Load SciSpaCy model + UMLS linker
# -----------------------------
nlp = spacy.load("en_core_sci_scibert")

if "scispacy_linker" not in nlp.pipe_names:
    nlp.add_pipe(
        "scispacy_linker",
        config={
            "linker_name": "umls",
            "resolve_abbreviations": True,
            "max_entities_per_mention": 1,
        },
    )

print("Loaded rows:", len(df))
print("Pipes:", nlp.pipe_names)

# -----------------------------
# Helpers
# -----------------------------
def cuis_from_doc(doc) -> set:
    cuis = set()
    for ent in doc.ents:
        if hasattr(ent._, "kb_ents"):
            for cui, score in ent._.kb_ents:
                cuis.add(cui)
    return cuis

def prf1(ref_set: set, pred_set: set):
    if len(ref_set) == 0 and len(pred_set) == 0:
        return 1.0, 1.0, 1.0
    if len(ref_set) == 0 or len(pred_set) == 0:
        return 0.0, 0.0, 0.0

    tp = len(ref_set & pred_set)
    p = tp / len(pred_set) if pred_set else 0.0
    r = tp / len(ref_set) if ref_set else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
    return p, r, f1

def extract_cui_sets(texts, batch_size=32):
    texts = pd.Series(texts).fillna("").astype(str).tolist()
    return [cuis_from_doc(doc) for doc in nlp.pipe(texts, batch_size=batch_size)]

def compare_concepts(df_in, col_ref, col_pred, name, batch_size=32):
    ref_sets = extract_cui_sets(df_in[col_ref], batch_size=batch_size)
    pred_sets = extract_cui_sets(df_in[col_pred], batch_size=batch_size)

    P, R, F1 = [], [], []
    TP = FP = FN = 0

    for rs, ps in zip(ref_sets, pred_sets):
        p, r, f = prf1(rs, ps)
        P.append(p)
        R.append(r)
        F1.append(f)

        TP += len(rs & ps)
        FP += len(ps - rs)
        FN += len(rs - ps)

    out = pd.DataFrame({
        f"{name}_ref_cuis": [";".join(sorted(s)) for s in ref_sets],
        f"{name}_pred_cuis": [";".join(sorted(s)) for s in pred_sets],
        f"{name}_P": P,
        f"{name}_R": R,
        f"{name}_F1": F1,
    })

    micro_P = TP / (TP + FP) if (TP + FP) else 0.0
    micro_R = TP / (TP + FN) if (TP + FN) else 0.0
    micro_F1 = (2 * micro_P * micro_R / (micro_P + micro_R)) if (micro_P + micro_R) else 0.0

    summary = {
        "model": "BioGPT",
        "comparison": name,
        "N": int(len(df_in)),
        "macro_P": float(np.mean(P)),
        "macro_R": float(np.mean(R)),
        "macro_F1": float(np.mean(F1)),
        "micro_P": float(micro_P),
        "micro_R": float(micro_R),
        "micro_F1": float(micro_F1),
        "TP": int(TP),
        "FP": int(FP),
        "FN": int(FN),
    }

    return out, summary

# -----------------------------
# RUN comparisons
# -----------------------------
# out_it, sum_it = compare_concepts(df, "input", "target", "input_vs_target")
out_tg, sum_tg = compare_concepts(df, "target", "prediction", "target_vs_generated")
# out_ig, sum_ig = compare_concepts(df, "input", "prediction", "input_vs_generated")

summary_df = pd.DataFrame([sum_tg])
summary_df

In [ ]:
# ---- Extra biomedical faithfulness diagnostics for BioGPT: target vs generated ----

input_sets  = extract_cui_sets(df["input"])
target_sets = extract_cui_sets(df["target"])
gen_sets    = extract_cui_sets(df["prediction"])

# Hallucinated concepts: generated CUIs not present in target/reference
halluc_counts = np.array([len(g - t) for t, g in zip(target_sets, gen_sets)])

# Omitted concepts: target/reference CUIs missing from generated output
omit_counts = np.array([len(t - g) for t, g in zip(target_sets, gen_sets)])

# Coverage: fraction of target/reference concepts recovered by generated output
coverage = np.array([
    (len(t & g) / len(t)) if len(t) else np.nan
    for t, g in zip(target_sets, gen_sets)
])

empty_input  = sum(len(s) == 0 for s in input_sets)
empty_target = sum(len(s) == 0 for s in target_sets)
empty_gen    = sum(len(s) == 0 for s in gen_sets)
empty_both_target_gen = sum((len(t) == 0 and len(g) == 0) for t, g in zip(target_sets, gen_sets))

print("Empty concept sets:")
print("  input empty:", empty_input)
print("  target empty:", empty_target)
print("  generated empty:", empty_gen)
print("  target & generated both empty:", empty_both_target_gen)

print("\nHallucination/Omission diagnostics:")
print("  Avg hallucinated CUIs (generated - target):", float(halluc_counts.mean()))
print("  Median hallucinated CUIs:", float(np.median(halluc_counts)))
print("  Avg omitted CUIs (target - generated):", float(omit_counts.mean()))
print("  Median omitted CUIs:", float(np.median(omit_counts)))

print("\nCoverage diagnostics:")
print("  Mean target coverage (|T∩G|/|T|):", float(np.nanmean(coverage)))
print("  Median target coverage:", float(np.nanmedian(coverage)))